# Exercise

Use the [typed-actors-nightmare](https://github.com/wilberquito/typed-actors-nightmare) repository to solve the following problem.

Feel free to add `objects`, `classes`, `traits` and test the functionalities in `*.worksheet.sc` files.

<span style="color: red;">**Note:** In Moodle you will find a deliverable task (4/5).</span>

## Distributed quicksort

In this assignment, you will explore **distributed sorting** of arrays using **actors**.

Remember the functional definition of `quicksort` (it is already implemented in the repository, you can use it).

```scala
def quicksortf(xs: Array[Int]): Array[Int] =
    if xs.length <= 1 then xs
    else
      val pivot: Int = xs(xs.length / 2)
      Array.concat(
        quicksortf(xs filter (x => pivot > x)), // define the whole lambda expression
        xs filter (_ == pivot), // use of lambda shorthand expression
        quicksortf(xs filter (pivot < _)) // use of lambda shorthand expression
      )


```

## Task

* Implement a **distributed sorting system** that sorts arrays of integers.
* You may use a **functional sorting algorithm** (like `quicksortf`) as a reference for local sorting.
* Extend it to a **distributed approach** using actors:

  * Small arrays (defined by a `limit`) can be sorted directly.
  * Larger arrays should be **split** and sorted concurrently by other actors.

## Exploration

* Experiment with different limits.

  * What happens if the `limit` is very low (such as 1)?
  * What happens if the `limit` is very high?
    
* Measure performance and consider the trade-offs between concurrency and overhead.


## Output

* Ensure your program can produce a **readable, verifiable output** of the sorted array.
    * You can tranform an array to a list using the `.toList` method. (lists are printable).
      

## Goals

* Practice **actor-based concurrency messaging**.
* Understand **distributed computation patterns**.
* Explore how task size affects **performance and efficiency**.
  

Follow the scheme:


```scala
import scala.concurrent.{Await, Future}
import scala.concurrent.duration.*
import scala.language.postfixOps
import akka.actor.{Actor, ActorLogging, ActorRef, ActorSystem, Props}
import akka.util.Timeout

import scala.util.Random
import scala.language.postfixOps


class Sorter(val limit: Int):

    . . .


object Main  {

  def main(args: Array[String]): Unit = {
    val max: Int = 1 << 14 // 2^14
    val arr: Array[Int] = (1 to max).toArray
    val shuffled: Array[Int] = Random.shuffle(arr).toArray
    val limit: Int = 1000

    val system: ActorSystem = . . .
    val sorter: ActorRef = . . .

    implicit val timeout: Timeout = Timeout(30.seconds)
    val t0: Long = System.nanoTime

    // Use ? to get a Future
    val future: Future[Any] = . . .
    val result: . . . = Await.result(future, timeout.duration).asInstanceOf[. . .]

    println(s"Main got reply: ")
    println(. . .)
    println(s"It took ${ (System.nanoTime - t0) / 1e9d } seconds")

    system.terminate()
  }
}

```


In [ ]:
// code goes here...

A good aproach...

In [ ]:
import akka.pattern.ask

import scala.concurrent.{Await, Future}
import scala.concurrent.duration.*
import scala.language.postfixOps
import akka.actor.{Actor, ActorLogging, ActorRef, ActorSystem, Props}
import akka.util.Timeout
import edu.udg.pda.sorting.FunctionalSortingAlgorithms.quicksortf

import scala.util.Random
import scala.language.postfixOps

trait ActorMessage

case class Sort(chunk: Array[Int]) extends ActorMessage
case class Sorted(chunk: Array[Int]) extends ActorMessage

class Sorter(val limit: Int) extends Actor with ActorLogging {
  def receive: Receive = {
    case Sort(chunk: Array[Int]) => {
      val replyTo = sender()
      if (chunk.length <= limit) {
        replyTo ! Sorted(quicksortf(chunk))
      }
      else {
        val pivot = chunk(chunk.length / 2)

        val leftWorker = context.actorOf(Props(new Sorter(limit)))
        val rightWorker = context.actorOf(Props(new Sorter(limit)))
        implicit val timeout: Timeout = Timeout(30.seconds)

        // Use ? to get a Future
        val leftFuture: Future[Any] = leftWorker ? Sort(chunk filter (pivot > _))
        val rightFuture: Future[Any] = rightWorker ? Sort(chunk filter (pivot < _))

        val leftResult: Sorted = Await.result(leftFuture, timeout.duration).asInstanceOf[Sorted]
        val rightResult: Sorted = Await.result(rightFuture, timeout.duration).asInstanceOf[Sorted]

        replyTo ! Sorted(
          Array.concat(
            leftResult.chunk,
            chunk filter (_ == pivot),
            rightResult.chunk)
        )
      }
    }
  }
}


object Main  {

  def main(args: Array[String]): Unit = {
    val max: Int = 1 << 14 // 2^14
    val arr: Array[Int] = (1 to max).toArray
    val shuffled: Array[Int] = Random.shuffle(arr).toArray
    val limit: Int = 1000

    val system: ActorSystem = ActorSystem("sort-system")
    val sorter: ActorRef = system.actorOf(Props(new Sorter(limit)), "sorter")

    implicit val timeout: Timeout = Timeout(30.seconds)

    val t0: Long = System.nanoTime
    
    // Use ? to get a Future
    val future: Future[Any] = sorter ? Sort(shuffled)
    val result: Sorted = Await.result(future, timeout.duration).asInstanceOf[Sorted]

    println(s"Main got reply: ")
    println(result.chunk.toList)

    println(s"It took ${ (System.nanoTime - t0) / 1e9d + " seconds" }")
    system.terminate()
  }
}


Can we do better?

In [ ]:
import akka.actor.{Actor, ActorLogging, ActorRef, ActorSystem, Props}
import akka.pattern.ask
import akka.util.Timeout

import scala.concurrent.duration.*
import scala.concurrent.{Await, Future}
import scala.language.postfixOps
import scala.util.Random

def quicksortf(xs: Array[Int]): Array[Int] =
  if xs.length <= 1 then
    xs
  else
    val pivot = xs(xs.length / 2)
    Array.concat(
      quicksortf(xs filter (x => pivot > x)), // Define the whole lambda expression
      xs filter (_ == pivot), // Use of lambda short-hand expression
      quicksortf(xs filter (pivot < _)) // Use of lambda short-hand expression
    )

trait ActorMessage

trait ChunkPart
trait Chunk

object LeftChunk extends ChunkPart
object RightChunk extends ChunkPart

case class Sort(chunk: Array[Int], kind: Option[ChunkPart]) extends ActorMessage
case class Sorted(chunk: Array[Int], kind: Option[ChunkPart]) extends ActorMessage

class Sorter(val limit: Int) extends Actor with ActorLogging {
  private var replyTo: ActorRef = null
  private var waiting: Int = 0
  private var leftChunk: Array[Int] = Array()
  private var rightChunk: Array[Int] = Array()
  private var splitChunk: Array[Int] = Array()
  private var selfKind: Option[ChunkPart] = None

  def receive: Receive = {
    case Sort(chunk: Array[Int], kind) =>
      replyTo = sender()
      selfKind = kind
      if (chunk.length <= limit) {
        replyTo ! Sorted(quicksortf(chunk), selfKind)
      }
      else {
        val pivot = chunk(chunk.length / 2)

        val leftWorker = context.actorOf(Props(new Sorter(limit)))
        val rightWorker = context.actorOf(Props(new Sorter(limit)))

        waiting = 2
        splitChunk = chunk filter (pivot == _)
        leftWorker ! Sort(chunk filter (pivot > _), Some(LeftChunk))
        rightWorker ! Sort(chunk filter (pivot < _), Some(RightChunk))
      }

    case Sorted(chunk, kind) =>
      kind match {
        case Some(s) =>
          s match {
            case LeftChunk => leftChunk = chunk
            case RightChunk => rightChunk = chunk
          }
          waiting -= 1
          if (waiting == 0) {
            val sorted: Array[Int] = Array.concat(leftChunk, splitChunk, rightChunk)
            replyTo ! Sorted(sorted, this.selfKind)
          }
        case None =>
          replyTo ! Sorted(chunk, None)
      }
  }
}


object Main  {
  def main(args: Array[String]): Unit = {
    val max: Int = 1 << 20 // 2^14
    val arr: Array[Int] = (1 to max).toArray
    val shuffled: Array[Int] = Random.shuffle(arr).toArray
    val limit: Int = 100000

    val system: ActorSystem = ActorSystem("sort-system")
    val sorter: ActorRef = system.actorOf(Props(new Sorter(limit)), "sorter")

    implicit val timeout: Timeout = Timeout(30.seconds)

    val t0: Long = System.nanoTime

    // Use ? to get a Future
    val future: Future[Any] = sorter ? Sort(shuffled, None)
    val result: Sorted = Await.result(future, timeout.duration).asInstanceOf[Sorted]

    println(s"Main got reply: ")
    println(result.chunk.toList)

    println(s"It took ${ (System.nanoTime - t0) / 1e9d + " seconds" }")
    system.terminate()
  }
}
